# Getting Started with Parallax

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/parallax/blob/main/docs/getting_started.ipynb)

This guide shows you how to shard a Flax NNX model with auto-sharding and how
to run on multi-host clusters. For an introduction to Parallax and its
features, see the [Parallax README](https://github.com/google/parallax#readme).


## Install Parallax

Install the latest release of Parallax from
[PyPI](https://pypi.org/project/google-parallax/):


In [ ]:
%pip install -q google-parallax

## Shard a Flax NNX model

After installing Parallax, you can use it to shard
[Flax NNX](https://flax.readthedocs.io/en/latest/index.html) models. The
following example uses auto-sharding on a 4 × 2 device mesh, so it needs 8
devices.

> **Note:** To run the example on a CPU-only machine (such as the default
> Colab runtime), simulate 8 devices by adding
> `jax.config.update('jax_num_cpu_devices', 8)` right after `import jax`.
> This must run before JAX initializes its backend, so restart the runtime if
> you have already run JAX code in this session.


In [ ]:
import jax

# Simulate 8 CPU devices. Remove this line when running on a real
# multi-device TPU or GPU host.
jax.config.update('jax_num_cpu_devices', 8)

import jax.numpy as jnp
from flax import nnx

import parallax

print(f'Devices: {jax.device_count()}')

In [ ]:
class Model(nnx.Module):

  def __init__(self, rngs: nnx.Rngs):
    self.linear1 = nnx.Linear(32, 128, rngs=rngs)
    self.linear2 = nnx.Linear(128, 10, rngs=rngs)

  def __call__(self, x):
    return self.linear2(nnx.relu(self.linear1(x)))


# Auto-sharding requires a 2D mesh. The axis names must match the
# `data_axis_name` and `model_axis_name` arguments below.
mesh = jax.make_mesh(
    (4, 2),
    ('x', 'y'),
    axis_types=(jax.sharding.AxisType.Auto,) * 2,
)

with jax.set_mesh(mesh):
  model = parallax.create_sharded_model(
      model_or_fn=lambda: Model(rngs=nnx.Rngs(0)),
      sample_inputs=(jnp.ones((1, 32)),),
      strategy=parallax.ShardingStrategy.AUTO,
      data_axis_name='x',
      model_axis_name='y',
  )
  outputs = model(jnp.ones((8, 32)))

print(outputs.shape)  # (8, 10)

In this example:

*   [`jax.make_mesh`](https://docs.jax.dev/en/latest/_autosummary/jax.make_mesh.html)
    creates a 2D mesh with a data axis (`'x'`) of size 4 and a model axis
    (`'y'`) of size 2, and
    [`jax.set_mesh`](https://docs.jax.dev/en/latest/_autosummary/jax.set_mesh.html)
    makes it the active mesh. `data_axis_name` and `model_axis_name` tell
    Parallax which axis is which.
*   `axis_types` sets both axes to `Auto`, so the compiler chooses shardings
    for intermediate values. With the default
    [`Explicit` axes](https://docs.jax.dev/en/latest/201/sharding.html#explicit-sharding-mode-makes-sharding-queryable-at-trace-time),
    calling this model raises a sharding error.
*   `model_or_fn` constructs the model. Parallax calls it so that parameters
    are created directly with the chosen shardings.
*   `sample_inputs` are used only to trace the model, so their batch size can
    differ from the inputs that you pass later.

You can inspect the shardings that Parallax chose for each parameter:


In [ ]:
for path, variable in nnx.iter_graph(model):
  if isinstance(variable, nnx.Variable):
    print('/'.join(map(str, path)), variable.sharding.spec)

To scale to more devices, change only the mesh shape. For example, `(2, 8)`
uses 16 devices. Your model code stays the same.

To use a sharding recipe instead of auto-sharding, set `strategy` to
[`parallax.ShardingStrategy.DDP`](https://github.com/google/parallax/blob/main/parallax/sharding/ddp.py)
or
[`parallax.ShardingStrategy.FSDP`](https://github.com/google/parallax/blob/main/parallax/sharding/fsdp.py).
For details, see
[Sharding Recipes](https://github.com/google/parallax/tree/main/parallax/sharding).


## Run on multi-host clusters

The sharding strategies (`AUTO`, `DDP`, and `FSDP`) build on the JAX
multi-process runtime. To run on multi-host TPU or GPU slices, run the same
Parallax program on every host, and call
[`jax.distributed.initialize()`](https://docs.jax.dev/en/latest/_autosummary/jax.distributed.initialize.html)
at the start of the program, before any other JAX calls:

```python
import jax

jax.distributed.initialize()
```

For details, see the JAX
[multi-process guide](https://docs.jax.dev/en/latest/501/multiprocess.html).


## What's next

*   Learn how Parallax chooses shardings in
    [AutoSharding](https://github.com/google/parallax/blob/main/parallax/sharding/auto_shard.py).
*   Apply Distributed Data Parallel
    ([DDP](https://github.com/google/parallax/blob/main/parallax/sharding/ddp.py))
    or Fully Sharded Data Parallel
    ([FSDP](https://github.com/google/parallax/blob/main/parallax/sharding/fsdp.py))
    with
    [Sharding Recipes](https://github.com/google/parallax/tree/main/parallax/sharding).
*   Reduce high-bandwidth memory (HBM) usage with
    [Model Offloading](https://github.com/google/parallax/blob/main/parallax/offload.py)
    and
    [Rematerialization](https://github.com/google/parallax/blob/main/parallax/offload.py).
